In [1]:
import pandas as pd
import numpy as np

# --- Load the filtered catalogue subset ---
df = pd.read_csv("filtered_events.csv")

# --- Classify each event into its observing run (O3 vs O4) from its name ---
# GW19* and GW20* (up to O3b, ending March 2020) -> O3
# GW23*, GW24*, GW25*                             -> O4
def classify_run(name: str) -> str:
    yy = name.replace("GW", "")[:2]
    if yy in ("19", "20"):
        return "O3"
    elif yy in ("23", "24", "25"):
        return "O4"
    else:
        return "unknown"

df["run"] = df["name"].apply(classify_run)

# --- Detector-frame chirp mass ---
# GWOSC convention: 'chirp_mass' (when populated) is already detector-frame.
# Where it's missing, derive it from the source-frame chirp mass and redshift:
#   M_det = M_source * (1 + z)
df["chirp_mass_det"] = df["chirp_mass"]
needs_derivation = df["chirp_mass_det"].isna()
df.loc[needs_derivation, "chirp_mass_det"] = (
    df.loc[needs_derivation, "chirp_mass_source"]
    * (1 + df.loc[needs_derivation, "redshift"])
)

# Flag any events for which neither route worked (e.g. missing redshift)
missing = df[df["chirp_mass_det"].isna()]
if not missing.empty:
    print(f"WARNING: could not compute detector-frame chirp mass for "
          f"{len(missing)} event(s): {missing['name'].tolist()}")

# --- Split into O3 / O4 dataframes ---
cols = ["name", "run", "chirp_mass_source", "redshift", "chirp_mass_det"]
o3_df = df[df["run"] == "O3"][cols].reset_index(drop=True)
o4_df = df[df["run"] == "O4"][cols].reset_index(drop=True)

print(f"O3 events: {len(o3_df)}")
display(o3_df)

print(f"\nO4 events: {len(o4_df)}")
display(o4_df)

O3 events: 13


,name,run,chirp_mass_source,redshift,chirp_mass_det
0,GW190412,O3,13.3,0.15,15.200
1,GW190828_063405,O3,25.0,0.41,35.000
2,GW190513_205428,O3,23.0,0.42,32.000
3,GW190706_222641,O3,42.0,0.80,80.000
4,GW190521_074359,O3,33.0,0.21,40.000
5,GW190519_153544,O3,43.0,0.50,70.000
6,GW190602_175927,O3,49.0,0.50,80.000
7,GW190408_181802,O3,18.0,0.30,24.000
8,GW190727_060333,O3,28.0,0.60,45.000
9,GW200129_065458,O3,27.2,0.18,32.096



O4 events: 33


,name,run,chirp_mass_source,redshift,chirp_mass_det
0,GW231226_101520,O4,32.50,0.23,39.9750
1,GW230927_153832,O4,16.37,0.23,20.1351
2,GW231123_135430,O4,101.00,0.40,141.4000
3,GW230919_215712,O4,21.00,0.25,26.2500
4,GW230627_015337,O4,6.02,0.07,6.4414
5,GW230628_231200,O4,25.50,0.40,35.7000
6,GW231028_153006,O4,63.00,0.68,105.8400
7,GW231102_071736,O4,43.30,0.63,70.5790
8,GW230914_111401,O4,39.80,0.47,58.5060
9,GW230924_124453,O4,22.30,0.42,31.6660


In [4]:
# --- Min/max detector-frame chirp mass per run ---
for run_name, run_df in [("O3", o3_df), ("O4", o4_df)]:
    valid = run_df.dropna(subset=["chirp_mass_det"])
    max_row = valid.loc[valid["chirp_mass_det"].idxmax()]
    min_row = valid.loc[valid["chirp_mass_det"].idxmin()]
    print(f"\n{run_name} detector-frame chirp mass range:")
    print(f"  Max: {max_row['chirp_mass_det']:.2f} Msun  ({max_row['name']})")
    print(f"  Min: {min_row['chirp_mass_det']:.2f} Msun  ({min_row['name']})")

# --- Min/max luminosity distance per run ---
for run_name, run_df in [("O3", o3_df), ("O4", o4_df)]:
    valid = run_df.dropna(subset=["redshift"])
    max_row = valid.loc[valid["redshift"].idxmax()]
    min_row = valid.loc[valid["redshift"].idxmin()]
    #convert redshift to luminosity distance using astropy.cosmology
    from astropy.cosmology import Planck18 as cosmo
    max_row["luminosity_distance"] = cosmo.luminosity_distance(max_row["redshift"]).value
    min_row["luminosity_distance"] = cosmo.luminosity_distance(min_row["redshift"]).value
    print(f"\n{run_name} redshift range:")
    print(f"  Max: {max_row['redshift']:.3f}  ({max_row['name']})")
    print(f"  Min: {min_row['redshift']:.3f}  ({min_row['name']})")
    print(f"\n{run_name} luminosity distance range:")
    print(f"  Max: {max_row['luminosity_distance']:.2f} Mpc  ({max_row['name']})")
    print(f"  Min: {min_row['luminosity_distance']:.2f} Mpc  ({min_row['name']})")


O3 detector-frame chirp mass range:
  Max: 80.00 Msun  (GW190706_222641)
  Min: 15.20 Msun  (GW190412)

O4 detector-frame chirp mass range:
  Max: 141.40 Msun  (GW231123_135430)
  Min: 6.44 Msun  (GW230627_015337)

O3 redshift range:
  Max: 0.800  (GW190706_222641)
  Min: 0.150  (GW190412)

O3 luminosity distance range:
  Max: 5162.17 Mpc  (GW190706_222641)
  Min: 736.92 Mpc  (GW190412)

O4 redshift range:
  Max: 0.960  (GW230922_040658)
  Min: 0.070  (GW230627_015337)

O4 luminosity distance range:
  Max: 6458.29 Mpc  (GW230922_040658)
  Min: 326.38 Mpc  (GW230627_015337)


/tmp/ipykernel_740392/668297910.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  max_row["luminosity_distance"] = cosmo.luminosity_distance(max_row["redshift"]).value
/tmp/ipykernel_740392/668297910.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  max_row["luminosity_distance"] = cosmo.luminosity_distance(max_row["redshift"]).value
/tmp/ipykernel_740392/668297910.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  min_row["lum

In [5]:
# --- Detector-frame individual component masses ---
# The catalogue only provides source-frame mass_1/mass_2, so derive
# detector-frame masses the same way: m_det = m_source * (1 + z)
df["mass_1_det"] = df["mass_1_source"] * (1 + df["redshift"])
df["mass_2_det"] = df["mass_2_source"] * (1 + df["redshift"])

# Refresh O3/O4 split to include the new columns
cols_indiv = ["name", "run", "mass_1_source", "mass_2_source", "redshift",
              "mass_1_det", "mass_2_det"]
o3_df_indiv = df[df["run"] == "O3"][cols_indiv].reset_index(drop=True)
o4_df_indiv = df[df["run"] == "O4"][cols_indiv].reset_index(drop=True)

print(f"O3 events: {len(o3_df_indiv)}")
display(o3_df_indiv)

print(f"\nO4 events: {len(o4_df_indiv)}")
display(o4_df_indiv)

# --- Min/max detector-frame individual mass per run ---
# Combine mass_1_det and mass_2_det into one pool per run, since "mass range"
# spans both companions, not just the primary.
for run_name, run_df in [("O3", o3_df_indiv), ("O4", o4_df_indiv)]:
    pooled = pd.concat([
        run_df[["name", "mass_1_det"]].rename(columns={"mass_1_det": "mass_det"}),
        run_df[["name", "mass_2_det"]].rename(columns={"mass_2_det": "mass_det"}),
    ]).dropna(subset=["mass_det"]).reset_index(drop=True)

    max_row = pooled.loc[pooled["mass_det"].idxmax()]
    min_row = pooled.loc[pooled["mass_det"].idxmin()]
    print(f"\n{run_name} detector-frame individual mass range (across mass_1 & mass_2):")
    print(f"  Max: {max_row['mass_det']:.2f} Msun  ({max_row['name']})")
    print(f"  Min: {min_row['mass_det']:.2f} Msun  ({min_row['name']})")

O3 events: 13


,name,run,mass_1_source,mass_2_source,redshift,mass_1_det,mass_2_det
0,GW190412,O3,31.0,8.0,0.15,35.650,9.200
1,GW190828_063405,O3,33.0,26.0,0.41,46.530,36.660
2,GW190513_205428,O3,41.0,18.0,0.42,58.220,25.560
3,GW190706_222641,O3,67.0,38.0,0.80,120.600,68.400
4,GW190521_074359,O3,43.0,35.0,0.21,52.030,42.350
5,GW190519_153544,O3,62.0,41.0,0.50,93.000,61.500
6,GW190602_175927,O3,73.0,44.0,0.50,109.500,66.000
7,GW190408_181802,O3,25.0,18.0,0.30,32.500,23.400
8,GW190727_060333,O3,38.0,29.0,0.60,60.800,46.400
9,GW200129_065458,O3,34.5,29.0,0.18,40.710,34.220



O4 events: 33


,name,run,mass_1_source,mass_2_source,redshift,mass_1_det,mass_2_det
0,GW231226_101520,O4,40.20,35.10,0.23,49.4460,43.1730
1,GW230927_153832,O4,21.70,16.60,0.23,26.6910,20.4180
2,GW231123_135430,O4,137.00,101.00,0.40,191.8000,141.4000
3,GW230919_215712,O4,27.30,21.40,0.25,34.1250,26.7500
4,GW230627_015337,O4,8.40,5.74,0.07,8.9880,6.1418
5,GW230628_231200,O4,32.50,27.00,0.40,45.5000,37.8000
6,GW231028_153006,O4,94.00,59.00,0.68,157.9200,99.1200
7,GW231102_071736,O4,61.00,42.00,0.63,99.4300,68.4600
8,GW230914_111401,O4,60.00,37.00,0.47,88.2000,54.3900
9,GW230924_124453,O4,28.80,23.20,0.42,40.8960,32.9440



O3 detector-frame individual mass range (across mass_1 & mass_2):
  Max: 120.60 Msun  (GW190706_222641)
  Min: 9.20 Msun  (GW190412)

O4 detector-frame individual mass range (across mass_1 & mass_2):
  Max: 191.80 Msun  (GW231123_135430)
  Min: 6.14 Msun  (GW230627_015337)
